In [1]:
# ============================================================
# MACHINE 3 RESUME - DIRECT PATH VERSION
# Resume rank4 + rank5 sequentially, safe mode
# ============================================================

# =========================
# 0. CONFIG
# =========================

WORLD_SIZE = 6
RESUME_RANKS = [4, 5]

# Exact old CSV paths from user
OLD_RANK4_CSV = "/kaggle/input/notebooks/thuhahaha/machine-3-imageclef-caption-qwen3-medgemma-hybrid/qwen3_full_6gpu/rank4_of_6/qwen3_full_candidates_rank4_of_6.csv"
OLD_RANK5_CSV = "/kaggle/input/notebooks/thuhahaha/machine-3-imageclef-caption-qwen3-medgemma-hybrid/qwen3_full_6gpu/rank5_of_6/qwen3_full_candidates_rank5_of_6.csv"

MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"

TEST_IMAGE_ROOT = "/kaggle/input/datasets/tanduy270206/clef-test-dataset/TEST DATASET/images"
CUI_SUBMISSION_CSV = "/kaggle/input/datasets/tanduy270206/submission-sinh-cuis/submission.csv"
CUI2META_JSON = "/kaggle/input/notebooks/tanduy270206/g-p-3-file-umls/cui2meta.json"

BASE_OUT_DIR = "/kaggle/working/qwen3_full_6gpu"

MAX_NEW_TOKENS = 80
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 512 * 28 * 28

# Resume ít ảnh nên save sau mỗi ảnh
SAVE_EVERY = 1

# Safe mode: dùng 1 GPU để resume tuần tự rank4 rồi rank5
CUDA_VISIBLE_DEVICE = "0"

# Nếu bị OOM, giảm:
# MAX_PIXELS = 448 * 28 * 28

# =========================
# 1. INSTALL
# =========================

import os
import sys
import subprocess
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICE
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def sh(cmd):
    print(f"\n[CMD] {cmd}")
    subprocess.check_call(cmd, shell=True)

print("=" * 100)
print("[STEP 1] Install dependencies")
print("=" * 100)

sh(
    f'{sys.executable} -m pip install -q --no-cache-dir --force-reinstall '
    f'"numpy==2.0.2" '
    f'"pandas==2.2.2" '
    f'"pillow==11.3.0" '
    f'"scipy==1.15.3" '
    f'"scikit-learn==1.6.1"'
)

sh(
    f'{sys.executable} -m pip install -q --no-cache-dir --upgrade '
    f'"accelerate>=1.2.0" '
    f'"bitsandbytes>=0.43.3" '
    f'"sentencepiece" '
    f'"tqdm" '
    f'"qwen-vl-utils==0.0.14" '
    f'"huggingface-hub>=1.5.0,<2.0" '
    f'"safetensors>=0.4.5"'
)

sh(
    f'{sys.executable} -m pip install -q --no-cache-dir '
    f'git+https://github.com/huggingface/transformers'
)

sh(
    f'{sys.executable} -m pip install -q --no-cache-dir '
    f'"qwen-vl-utils==0.0.14"'
)

# =========================
# 2. IMPORTS
# =========================

print("=" * 100)
print("[STEP 2] Imports")
print("=" * 100)

import re
import gc
import json
import time
import zipfile
import shutil
import warnings

import numpy as np
import pandas as pd
import PIL
import scipy
import sklearn
import torch
import transformers
import huggingface_hub

from tqdm.auto import tqdm
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

warnings.filterwarnings("ignore")

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("Pillow:", PIL.__version__)
print("scipy:", scipy.__version__)
print("sklearn:", sklearn.__version__)
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Visible GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Visible GPU name:", torch.cuda.get_device_name(0))

# =========================
# 3. PATH SETUP + COPY OLD CSV
# =========================

print("=" * 100)
print("[STEP 3] Restore old rank4/rank5 CSV")
print("=" * 100)

OLD_RANK4_CSV = Path(OLD_RANK4_CSV)
OLD_RANK5_CSV = Path(OLD_RANK5_CSV)

TEST_IMAGE_ROOT = Path(TEST_IMAGE_ROOT)
if not TEST_IMAGE_ROOT.exists():
    TEST_IMAGE_ROOT = Path("/kaggle/input/datasets/tanduy270206/clef-test-dataset/TEST DATASET")

CUI_SUBMISSION_CSV = Path(CUI_SUBMISSION_CSV)
CUI2META_JSON = Path(CUI2META_JSON)

BASE_OUT_DIR = Path(BASE_OUT_DIR)

rank4_dst = BASE_OUT_DIR / "rank4_of_6" / "qwen3_full_candidates_rank4_of_6.csv"
rank5_dst = BASE_OUT_DIR / "rank5_of_6" / "qwen3_full_candidates_rank5_of_6.csv"

rank4_dst.parent.mkdir(parents=True, exist_ok=True)
rank5_dst.parent.mkdir(parents=True, exist_ok=True)

print("OLD_RANK4_CSV:", OLD_RANK4_CSV, OLD_RANK4_CSV.exists())
print("OLD_RANK5_CSV:", OLD_RANK5_CSV, OLD_RANK5_CSV.exists())
print("TEST_IMAGE_ROOT:", TEST_IMAGE_ROOT, TEST_IMAGE_ROOT.exists())
print("CUI_SUBMISSION_CSV:", CUI_SUBMISSION_CSV, CUI_SUBMISSION_CSV.exists())
print("CUI2META_JSON:", CUI2META_JSON, CUI2META_JSON.exists())

if not OLD_RANK4_CSV.exists():
    raise FileNotFoundError(f"Missing OLD_RANK4_CSV: {OLD_RANK4_CSV}")
if not OLD_RANK5_CSV.exists():
    raise FileNotFoundError(f"Missing OLD_RANK5_CSV: {OLD_RANK5_CSV}")
if not TEST_IMAGE_ROOT.exists():
    raise FileNotFoundError(f"Missing TEST_IMAGE_ROOT: {TEST_IMAGE_ROOT}")
if not CUI_SUBMISSION_CSV.exists():
    raise FileNotFoundError(f"Missing CUI_SUBMISSION_CSV: {CUI_SUBMISSION_CSV}")
if not CUI2META_JSON.exists():
    raise FileNotFoundError(f"Missing CUI2META_JSON: {CUI2META_JSON}")

shutil.copy2(OLD_RANK4_CSV, rank4_dst)
shutil.copy2(OLD_RANK5_CSV, rank5_dst)

print("Copied rank4 ->", rank4_dst)
print("Copied rank5 ->", rank5_dst)

rank_csv_map = {
    4: rank4_dst,
    5: rank5_dst,
}

# =========================
# 4. SCAN TEST IMAGES
# =========================

print("=" * 100)
print("[STEP 4] Scan test images")
print("=" * 100)

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

def numeric_suffix_value(x):
    s = str(x)
    m = re.search(r"(\d+)(?!.*\d)", s)
    return int(m.group(1)) if m else 10**18

def numeric_suffix_key(series):
    return series.map(numeric_suffix_value)

image_paths = []
for p in TEST_IMAGE_ROOT.rglob("*"):
    if p.suffix.lower() in IMG_EXTS:
        image_paths.append(p)

df_images = pd.DataFrame({
    "ID": [p.stem for p in image_paths],
    "path": [str(p) for p in image_paths],
})

df_images["ID"] = df_images["ID"].astype(str)
df_images = df_images.sort_values("ID", key=numeric_suffix_key).reset_index(drop=True)

print("Total test images:", len(df_images))

# =========================
# 5. LOAD CUI
# =========================

print("=" * 100)
print("[STEP 5] Load CUI")
print("=" * 100)

df_cui = pd.read_csv(CUI_SUBMISSION_CSV)

if "id" in df_cui.columns and "ID" not in df_cui.columns:
    df_cui = df_cui.rename(columns={"id": "ID"})

if "CUIs" not in df_cui.columns:
    possible = [c for c in df_cui.columns if c.lower() in ["cui", "cuis", "concept", "concepts"]]
    if possible:
        df_cui = df_cui.rename(columns={possible[0]: "CUIs"})
    else:
        df_cui["CUIs"] = ""

df_cui["ID"] = df_cui["ID"].astype(str)
df_cui["CUIs"] = df_cui["CUIs"].fillna("").astype(str)

with open(CUI2META_JSON, "r", encoding="utf-8") as f:
    cui2meta = json.load(f)

def cui_to_terms(cui_string, max_terms=8):
    if pd.isna(cui_string):
        return ""

    raw = str(cui_string).strip()
    if not raw:
        return ""

    cuis = re.split(r"[|,; \t\n]+", raw)
    terms = []

    for cui in cuis:
        cui = cui.strip()
        if not cui:
            continue

        meta = cui2meta.get(cui, {})

        if isinstance(meta, dict):
            term = (
                meta.get("term")
                or meta.get("name")
                or meta.get("preferred_name")
                or meta.get("label")
                or ""
            )
        else:
            term = str(meta)

        term = str(term).strip()

        if term and term.lower() not in [t.lower() for t in terms]:
            terms.append(term)

        if len(terms) >= max_terms:
            break

    return "; ".join(terms)

df_cui["cui_terms"] = df_cui["CUIs"].apply(cui_to_terms)

# =========================
# 6. CLEANER + PROMPT
# =========================

BAD_PREFIXES = [
    r"^caption\s*:\s*",
    r"^answer\s*:\s*",
    r"^output\s*:\s*",
    r"^final caption\s*:\s*",
    r"^this image shows\s+",
    r"^the image shows\s+",
    r"^this figure shows\s+",
    r"^the figure shows\s+",
    r"^an image of\s+",
    r"^a medical image of\s+",
]

GENERIC_BAD = {
    "the relevant anatomical region",
    "relevant anatomical region",
    "medical image showing findings",
    "the image shows the relevant anatomy",
    "an image of the relevant anatomical region",
    "medical imaging figure",
    "medical image",
}

MODALITY_REPLACEMENTS = [
    (r"\bX-Ray Computed Tomography\b", "CT"),
    (r"\bComputed Tomography\b", "CT"),
    (r"\bcomputed tomography\b", "CT"),
    (r"\bComputerized Tomography\b", "CT"),
    (r"\bMagnetic Resonance Imaging\b", "MRI"),
    (r"\bmagnetic resonance imaging\b", "MRI"),
    (r"\bUltrasonography\b", "ultrasound"),
    (r"\bultrasonography\b", "ultrasound"),
    (r"\bSonography\b", "ultrasound"),
    (r"\bsonography\b", "ultrasound"),
    (r"\bPlain x-ray\b", "X-ray"),
    (r"\bplain x-ray\b", "X-ray"),
    (r"\bplain radiograph\b", "X-ray"),
]

TRUNCATED_ENDINGS = [
    r"\blikely\.$",
    r"\bsuggestive of\.$",
    r"\bindicative of\.$",
    r"\bconsistent with\.$",
    r"\bwith no\.$",
    r"\blocated in the\.$",
    r"\bat the\.$",
    r"\bdemonstrating\.$",
    r"\bshowing\.$",
]

def clean_caption_v4(text):
    if pd.isna(text):
        return ""

    s = str(text).strip()
    s = s.replace("\n", " ")
    s = re.sub(r"\s+", " ", s).strip()
    s = s.strip("`").strip()
    s = s.strip("\"'“”‘’").strip()

    for pat in BAD_PREFIXES:
        s = re.sub(pat, "", s, flags=re.IGNORECASE).strip()

    s = re.sub(r"(?i)^write one concise.*?caption[:\s-]*", "", s).strip()
    s = re.sub(r"(?i)^generate one concise.*?caption[:\s-]*", "", s).strip()
    s = re.sub(r"(?i)^final caption[:\s-]*", "", s).strip()

    for pat, rep in MODALITY_REPLACEMENTS:
        s = re.sub(pat, rep, s, flags=re.IGNORECASE)

    s = re.sub(r"\s+([,.;:!?])", r"\1", s)
    s = re.sub(r"\s+", " ", s).strip()

    words = s.split()
    if len(words) > 55:
        sentences = re.split(r"(?<=[.!?])\s+", s)
        if len(sentences) > 1:
            s = sentences[0].strip()
            if len(s.split()) < 8 and len(sentences) > 1:
                s = (s + " " + sentences[1]).strip()
        else:
            s = " ".join(words[:55]).strip()

    for pat in TRUNCATED_ENDINGS:
        if re.search(pat, s, flags=re.IGNORECASE):
            s = re.sub(pat, ".", s, flags=re.IGNORECASE).strip()

    s = s.strip()

    if s.lower().strip(". ") in GENERIC_BAD:
        return ""

    if s and s[-1] not in ".!?":
        s += "."

    return s

def build_qwen3_prompt(cui_terms):
    hint = ""

    if cui_terms and len(str(cui_terms).strip()) > 0:
        hint = f"""
Possible UMLS/CUI clinical hints: {cui_terms}
Use these hints only if they are visually supported by the image.
Do not force unsupported diseases or findings.
"""

    return f"""
You are generating a caption for a medical figure in a biomedical article.

Task:
Write one concise, clinically relevant English caption for the image.

Rules:
- Focus on the visible medical content.
- Mention modality when clear: CT, MRI, X-ray, ultrasound, histology, endoscopy, angiography, PET/CT, microscopy.
- Mention anatomical region and main visual finding if visible.
- Do not write "This image shows".
- Do not write "Caption:".
- Do not include uncertainty unless needed.
- Do not invent patient age, sex, diagnosis, treatment, or outcome if not visible.
- Output only one caption sentence.

{hint}
Final caption:
""".strip()

# =========================
# 7. BUILD TARGETS + FIND MISSING
# =========================

print("=" * 100)
print("[STEP 7] Find missing rows for rank4/rank5")
print("=" * 100)

rank_targets = {}
missing_map = {}
old_map = {}

for rank in RESUME_RANKS:
    print("\n" + "=" * 80)
    print(f"Rank {rank}")

    df_images_rank = df_images.iloc[rank::WORLD_SIZE].reset_index(drop=True)

    df_rank = df_images_rank.merge(
        df_cui[["ID", "CUIs", "cui_terms"]],
        on="ID",
        how="left"
    )

    df_rank["CUIs"] = df_rank["CUIs"].fillna("")
    df_rank["cui_terms"] = df_rank["cui_terms"].fillna("")

    csv_path = rank_csv_map[rank]

    old = pd.read_csv(csv_path)
    old["ID"] = old["ID"].astype(str)
    old = old.drop_duplicates("ID", keep="first").reset_index(drop=True)
    old["caption_qwen3"] = old["caption_qwen3"].fillna("").astype(str)

    done_ids = set(old.loc[old["caption_qwen3"].str.strip() != "", "ID"])
    missing_df = df_rank[~df_rank["ID"].astype(str).isin(done_ids)].copy()

    rank_targets[rank] = df_rank
    missing_map[rank] = missing_df
    old_map[rank] = old

    print("CSV:", csv_path)
    print("Existing rows:", len(old))
    print("Target rows:", len(df_rank))
    print("Done valid rows:", len(done_ids))
    print("Missing rows to generate:", len(missing_df))
    print("Old empty:", old["caption_qwen3"].str.strip().eq("").sum())
    print("Old duplicate IDs:", old["ID"].duplicated().sum())

    if len(missing_df) > 0:
        print("Missing preview:")
        print(missing_df[["ID"]].head(20).to_string(index=False))

total_missing = sum(len(missing_map[r]) for r in RESUME_RANKS)
print("\nTOTAL MISSING:", total_missing)

# =========================
# 8. LOAD MODEL IF NEEDED
# =========================

print("=" * 100)
print("[STEP 8] Load model if needed")
print("=" * 100)

model = None
processor = None

if total_missing == 0:
    print("Both rank4 and rank5 already complete. No model loading needed.")
else:
    print("Loading model...")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )

    processor = AutoProcessor.from_pretrained(
        MODEL_ID,
        min_pixels=MIN_PIXELS,
        max_pixels=MAX_PIXELS,
        trust_remote_code=True,
    )

    model.eval()

    try:
        print("model.device:", model.device)
    except Exception:
        print("first param device:", next(model.parameters()).device)

@torch.inference_mode()
def generate_caption(image_path, cui_terms="", max_new_tokens=80):
    prompt = build_qwen3_prompt(cui_terms)

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": str(image_path),
                    "min_pixels": MIN_PIXELS,
                    "max_pixels": MAX_PIXELS,
                },
                {"type": "text", "text": prompt},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )

    try:
        inputs = inputs.to(model.device)
    except Exception:
        first_device = next(model.parameters()).device
        inputs = {
            k: v.to(first_device) if hasattr(v, "to") else v
            for k, v in inputs.items()
        }

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=True,
    )

    input_len = inputs["input_ids"].shape[-1]
    generated_ids_trimmed = generated_ids[:, input_len:]

    output_text = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return output_text.strip()

# =========================
# 9. RESUME RANK4 THEN RANK5
# =========================

print("=" * 100)
print("[STEP 9] Resume rank4 then rank5")
print("=" * 100)

for rank in RESUME_RANKS:
    missing_df = missing_map[rank]
    csv_path = rank_csv_map[rank]
    df_rank = rank_targets[rank]
    records = old_map[rank].to_dict("records")

    print("\n" + "=" * 100)
    print(f"RESUME RANK {rank}")
    print("Missing:", len(missing_df))
    print("=" * 100)

    if len(missing_df) == 0:
        print(f"Rank {rank} already complete.")
        continue

    start = time.time()

    for _, row in tqdm(missing_df.iterrows(), total=len(missing_df), desc=f"resume_rank{rank}"):
        img_id = str(row["ID"])

        try:
            t0 = time.time()
            raw = generate_caption(row["path"], row["cui_terms"], MAX_NEW_TOKENS)
            sec = time.time() - t0
            clean = clean_caption_v4(raw)

        except Exception as e:
            print("[ERROR]", img_id, repr(e))
            raw = ""
            clean = ""
            sec = -1

        records.append({
            "ID": img_id,
            "path": row["path"],
            "CUIs": row["CUIs"],
            "cui_terms": row["cui_terms"],
            "caption_qwen3_raw": raw,
            "caption_qwen3": clean,
            "seconds": sec,
        })

        # save after every image
        tmp = pd.DataFrame(records)
        tmp["ID"] = tmp["ID"].astype(str)
        tmp = tmp.drop_duplicates("ID", keep="first")
        tmp = tmp.sort_values("ID", key=numeric_suffix_key).reset_index(drop=True)
        tmp.to_csv(csv_path, index=False)

        print(f"Rank {rank} saved rows:", len(tmp), "/", len(df_rank))

    elapsed = (time.time() - start) / 60
    print(f"Rank {rank} resume elapsed minutes:", round(elapsed, 2))

# =========================
# 10. FINAL CHECK MACHINE 3
# =========================

print("=" * 100)
print("[STEP 10] Final check Machine 3")
print("=" * 100)

final_parts = []

for rank in RESUME_RANKS:
    csv_path = rank_csv_map[rank]
    df_rank = rank_targets[rank]

    final = pd.read_csv(csv_path)
    final["ID"] = final["ID"].astype(str)
    final = final.drop_duplicates("ID", keep="first")
    final = final.sort_values("ID", key=numeric_suffix_key).reset_index(drop=True)
    final.to_csv(csv_path, index=False)

    final_parts.append(final)

    print("\n" + "=" * 80)
    print(f"Rank {rank}")
    print("CSV:", csv_path)
    print("Rows:", len(final))
    print("Target:", len(df_rank))
    print("Empty:", final["caption_qwen3"].fillna("").astype(str).str.strip().eq("").sum())
    print("Duplicate IDs:", final["ID"].duplicated().sum())
    print("Unique captions:", final["caption_qwen3"].nunique())

machine3 = pd.concat(final_parts, ignore_index=True)

print("\n" + "=" * 80)
print("Machine 3 combined")
print("Rows:", len(machine3))
print("Empty:", machine3["caption_qwen3"].fillna("").astype(str).str.strip().eq("").sum())
print("Duplicate IDs:", machine3["ID"].astype(str).duplicated().sum())
print("Unique captions:", machine3["caption_qwen3"].nunique())

# =========================
# 11. ZIP OUTPUT
# =========================

print("=" * 100)
print("[STEP 11] Zip Machine 3 result")
print("=" * 100)

zip_out = Path("/kaggle/working/machine3_resume_result.zip")

with zipfile.ZipFile(zip_out, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(rank4_dst, arcname="qwen3_full_6gpu/rank4_of_6/qwen3_full_candidates_rank4_of_6.csv")
    z.write(rank5_dst, arcname="qwen3_full_6gpu/rank5_of_6/qwen3_full_candidates_rank5_of_6.csv")

print("Saved zip:", zip_out)
print("=" * 100)
print("DONE MACHINE 3 RESUME")
print("=" * 100)

[STEP 1] Install dependencies

[CMD] /usr/bin/python3 -m pip install -q --no-cache-dir --force-reinstall "numpy==2.0.2" "pandas==2.2.2" "pillow==11.3.0" "scipy==1.15.3" "scikit-learn==1.6.1"
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 175.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 174.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 180.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 172.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 176.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 kB 389.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 378.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.1/510.1 kB 389.7 MB/s eta 0:00:00
   ━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.



[CMD] /usr/bin/python3 -m pip install -q --no-cache-dir --upgrade "accelerate>=1.2.0" "bitsandbytes>=0.43.3" "sentencepiece" "tqdm" "qwen-vl-utils==0.0.14" "huggingface-hub>=1.5.0,<2.0" "safetensors>=0.4.5"
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 279.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 392.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 193.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 242.2 MB/s eta 0:00:00

[CMD] /usr/bin/python3 -m pip install -q --no-cache-dir git+https://github.com/huggingface/transformers

[CMD] /usr/bin/python3 -m pip install -q --no-cache-dir "qwen-vl-utils==0.0.14"
[STEP 2] Imports
numpy: 2.0.2
pandas: 2.2.2
Pillow: 11.3.0
scipy: 1.15.3
sklearn: 1.6.1
torch: 2.10.0+cu128
transformers: 5.8.0.dev0
huggingface_hub: 1.13.0
CUDA available: True
Visible GPU count: 1
Visible

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model.device: cuda:0
[STEP 9] Resume rank4 then rank5

RESUME RANK 4
Missing: 41


resume_rank4:   0%|          | 0/41 [00:00<?, ?it/s]

Rank 4 saved rows: 2501 / 2541
Rank 4 saved rows: 2502 / 2541
Rank 4 saved rows: 2503 / 2541
Rank 4 saved rows: 2504 / 2541
Rank 4 saved rows: 2505 / 2541
Rank 4 saved rows: 2506 / 2541
Rank 4 saved rows: 2507 / 2541
Rank 4 saved rows: 2508 / 2541
Rank 4 saved rows: 2509 / 2541
Rank 4 saved rows: 2510 / 2541
Rank 4 saved rows: 2511 / 2541
Rank 4 saved rows: 2512 / 2541
Rank 4 saved rows: 2513 / 2541
Rank 4 saved rows: 2514 / 2541
Rank 4 saved rows: 2515 / 2541
Rank 4 saved rows: 2516 / 2541
Rank 4 saved rows: 2517 / 2541
Rank 4 saved rows: 2518 / 2541
Rank 4 saved rows: 2519 / 2541
Rank 4 saved rows: 2520 / 2541
Rank 4 saved rows: 2521 / 2541
Rank 4 saved rows: 2522 / 2541
Rank 4 saved rows: 2523 / 2541
Rank 4 saved rows: 2524 / 2541
Rank 4 saved rows: 2525 / 2541
Rank 4 saved rows: 2526 / 2541
Rank 4 saved rows: 2527 / 2541
Rank 4 saved rows: 2528 / 2541
Rank 4 saved rows: 2529 / 2541
Rank 4 saved rows: 2530 / 2541
Rank 4 saved rows: 2531 / 2541
Rank 4 saved rows: 2532 / 2541
Rank 4 s

resume_rank5:   0%|          | 0/81 [00:00<?, ?it/s]

Rank 5 saved rows: 2461 / 2541
Rank 5 saved rows: 2462 / 2541
Rank 5 saved rows: 2463 / 2541
Rank 5 saved rows: 2464 / 2541
Rank 5 saved rows: 2465 / 2541
Rank 5 saved rows: 2466 / 2541
Rank 5 saved rows: 2467 / 2541
Rank 5 saved rows: 2468 / 2541
Rank 5 saved rows: 2469 / 2541
Rank 5 saved rows: 2470 / 2541
Rank 5 saved rows: 2471 / 2541
Rank 5 saved rows: 2472 / 2541
Rank 5 saved rows: 2473 / 2541
Rank 5 saved rows: 2474 / 2541
Rank 5 saved rows: 2475 / 2541
Rank 5 saved rows: 2476 / 2541
Rank 5 saved rows: 2477 / 2541
Rank 5 saved rows: 2478 / 2541
Rank 5 saved rows: 2479 / 2541
Rank 5 saved rows: 2480 / 2541
Rank 5 saved rows: 2481 / 2541
Rank 5 saved rows: 2482 / 2541
Rank 5 saved rows: 2483 / 2541
Rank 5 saved rows: 2484 / 2541
Rank 5 saved rows: 2485 / 2541
Rank 5 saved rows: 2486 / 2541
Rank 5 saved rows: 2487 / 2541
Rank 5 saved rows: 2488 / 2541
Rank 5 saved rows: 2489 / 2541
Rank 5 saved rows: 2490 / 2541
Rank 5 saved rows: 2491 / 2541
Rank 5 saved rows: 2492 / 2541
Rank 5 s